# Governance Mark Emails Sent

Runs as the **final stage** of the pipeline, after all Outlook activities have completed.

Marks every `pending` row in `email_outbox` as `sent` so the next pipeline run doesn't resend them.

### Why this is needed
The pipeline's Lookup activities filter on `status = 'pending'`. Without this step, every run
would resend all previously generated emails.

### Permission
**Contributor** — writes to a Delta table only. No API calls.


## Parameters

In [ ]:
# Set by the pipeline (or leave default for manual runs)
mark_status = "sent"   # or "failed" if you wire a failure path


## Mark pending emails

In [ ]:
import pandas as pd
import logging
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("mark_sent")

run_timestamp = datetime.now(timezone.utc).isoformat()

df_outbox = spark.sql("SELECT * FROM email_outbox").toPandas()
log.info(f"email_outbox total rows: {len(df_outbox)}")

if df_outbox.empty:
    log.info("Outbox is empty. Nothing to mark.")
else:
    pending_mask = df_outbox["status"] == "pending"
    pending_count = int(pending_mask.sum())
    log.info(f"Pending emails: {pending_count}")

    if pending_count == 0:
        log.info("No pending emails. Nothing to mark.")
    else:
        # Show what we're marking
        for etype, count in df_outbox.loc[pending_mask, "email_type"].value_counts().items():
            log.info(f"  {etype}: {count}")

        # Mark them
        df_outbox.loc[pending_mask, "status"] = mark_status

        # Add a sent_at column if it doesn't exist
        if "sent_at" not in df_outbox.columns:
            df_outbox["sent_at"] = None
        df_outbox.loc[pending_mask, "sent_at"] = run_timestamp

        spark.createDataFrame(df_outbox.astype(str)) \
            .write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("email_outbox")

        log.info(f"✔ Marked {pending_count} emails as '{mark_status}'")


## Summary

In [ ]:
df_final = spark.sql("SELECT status, email_type, COUNT(*) as count FROM email_outbox GROUP BY status, email_type ORDER BY status, email_type").toPandas()

print("=" * 55)
print("  EMAIL OUTBOX STATE")
print("=" * 55)
if df_final.empty:
    print("  Outbox is empty.")
else:
    for _, r in df_final.iterrows():
        print(f"  {r['status']:10s} {r['email_type']:28s} {r['count']}")
    print(f"  {'-' * 51}")
    print(f"  {'Total':10s} {'':28s} {df_final['count'].sum()}")
print("=" * 55)


## Optional: purge old sent emails

The outbox grows with every run. Each row holds a full HTML body (~20 KB), so it is worth
trimming periodically. Uncomment and run this to keep only the last 30 days.


In [ ]:
# from datetime import timedelta
#
# cutoff = (datetime.now(timezone.utc) - timedelta(days=30)).isoformat()
# df_ob = spark.sql("SELECT * FROM email_outbox").toPandas()
# before = len(df_ob)
# df_ob = df_ob[df_ob["created_at"] >= cutoff]
# spark.createDataFrame(df_ob.astype(str)).write.format("delta").mode("overwrite") \
#     .option("overwriteSchema", "true").saveAsTable("email_outbox")
# print(f"Purged {before - len(df_ob)} rows older than 30 days. Remaining: {len(df_ob)}")
